# DistilBERT

## Load train/val/test sets
In this step, I will load the files from the split_data folder to ensure that every model is trained, tuned and evaluated on exactly the same input texts and labels. To load the data, I will use the "load_split_data" function from the `helper.py` file:

In [34]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [35]:
from utilities import helper

X_train, X_val, X_test, y_train, y_val, y_test = helper.load_split_data(show = True)
labels = list(y_train.columns)

X_train: (12122,)
X_val: (3742,)
X_test: (3742,)
y_train: (12122, 5)
y_val: (3742, 5)
y_test: (3742, 5)


## Build hugging face dataframe per split

In [36]:
import numpy as np
from datasets import Dataset
import pandas as pd
from utilities.text_normalization import preprocess_text

def make_hf_split(X, y, label_cols):
    df = pd.concat([pd.Series(X, name="message").reset_index(drop=True),y.reset_index(drop=True)], axis=1)
    df["message"] = df["message"].apply(lambda t: preprocess_text(t, lemmatize=False))
    df[label_cols] = df[label_cols].astype(np.float32)
    df["labels"] = df[label_cols].values.tolist()
    df = df.drop(columns=label_cols)
    return Dataset.from_pandas(df)

train_ds = make_hf_split(X_train, y_train, labels)
val_ds   = make_hf_split(X_val, y_val, labels)
test_ds  = make_hf_split(X_test, y_test, labels)
train_ds

Dataset({
    features: ['message', 'labels'],
    num_rows: 12122
})

## Tokenize text

In [37]:
from transformers import AutoTokenizer

checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
def tok(batch):
    return tokenizer(batch["message"], truncation=True, max_length=128)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)
train_ds

Map:   0%|          | 0/12122 [00:00<?, ? examples/s]

Map:   0%|          | 0/3742 [00:00<?, ? examples/s]

Map:   0%|          | 0/3742 [00:00<?, ? examples/s]

Dataset({
    features: ['message', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 12122
})

## Set dataset format for PyTorch

In [38]:
cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)
train_ds

Dataset({
    features: ['message', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 12122
})

## Load model

In [39]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    problem_type="multi_label_classification",
)
print(model.config.num_labels)
print(model.config.problem_type)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


5
multi_label_classification


## Fine-tune model

In [40]:
import numpy as np
import torch
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score, hamming_loss, jaccard_score
from transformers import EarlyStoppingCallback, Trainer

data_collator = DataCollatorWithPadding(tokenizer)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming": hamming_loss(y_true, y_pred),
        "jaccard_micro": jaccard_score(y_true, y_pred, average="micro", zero_division=0),
    }

def tune_per_label_thresholds(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)

    L = y_true.shape[1]
    best_t = np.full(L, 0.5, dtype=float)
    best_f = np.zeros(L, dtype=float)

    for j in range(L):
        yj_true = y_true[:, j]
        if yj_true.sum() == 0:
            continue

        f1s = []
        for t in grid:
            yj_pred = (y_prob[:, j] >= t).astype(int)
            f1s.append(f1_score(yj_true, yj_pred, zero_division=0))

        k = int(np.argmax(f1s))
        best_t[j] = float(grid[k])
        best_f[j] = float(f1s[k])

    return best_t, best_f

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def eval_with_thresholds(logits, y_true, thresholds):
    probs = sigmoid(logits)
    y_pred = (probs >= thresholds).astype(int)

    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming": hamming_loss(y_true, y_pred),
        "jaccard_micro": jaccard_score(y_true, y_pred, average="micro", zero_division=0),
    }

args = TrainingArguments(
    output_dir="distilbert_crisis",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer.train()

val_pred = trainer.predict(val_ds)
val_logits = val_pred.predictions
val_true   = val_pred.label_ids
val_probs = sigmoid(val_logits)

best_t, best_f = tune_per_label_thresholds(val_true, val_probs)

print("Per-label tuned thresholds (VAL):")
for name, t, f in zip(labels, best_t, best_f):
    print(f"{name:25s}  best_t={t:.2f}  val_f1={f:.3f}")

print("\nVAL overall metrics using tuned thresholds:")
print(eval_with_thresholds(val_logits, val_true, best_t))

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Hamming,Jaccard Micro
1,0.230400,0.183413,0.675177,0.650951,0.063923,0.509635
2,0.208200,0.181561,0.677748,0.651169,0.064244,0.512571
3,0.155800,0.186932,0.679882,0.654022,0.063870,0.515016
4,0.152800,0.207540,0.677311,0.655278,0.066970,0.512072


Per-label tuned thresholds (VAL):
need_basic_supplies        best_t=0.65  val_f1=0.804
need_medical_help          best_t=0.25  val_f1=0.618
need_safety_rescue         best_t=0.40  val_f1=0.510
need_shelter               best_t=0.60  val_f1=0.743
people_status              best_t=0.45  val_f1=0.629

VAL overall metrics using tuned thresholds:
{'micro_f1': 0.6826597131681877, 'macro_f1': 0.6609246514482099, 'hamming': 0.06504543025120256, 'jaccard_micro': 0.5182106096595408}


In [41]:
test_pred = trainer.predict(test_ds)
test_logits = test_pred.predictions
test_true   = test_pred.label_ids
test_probs  = 1 / (1 + np.exp(-test_logits))
test_pred_bin = (test_probs >= best_t).astype(int)

test_metrics = eval_with_thresholds(test_logits, test_true, best_t)
row = {"model": "DistilBERT",**test_metrics}
results_df = pd.DataFrame([row])
results_df.to_csv("../data/results/distilbert_results.csv", index=False)
results_df

,model,micro_f1,macro_f1,hamming,jaccard_micro
0,DistilBERT,0.671279,0.644755,0.068573,0.505206


In [42]:
from sklearn.metrics import classification_report

print(classification_report(
    test_true, test_pred_bin,
    target_names=labels,
    zero_division=0
))

                     precision    recall  f1-score   support

need_basic_supplies       0.76      0.84      0.80       576
  need_medical_help       0.57      0.68      0.62       433
 need_safety_rescue       0.51      0.51      0.51       269
       need_shelter       0.75      0.68      0.71       355
      people_status       0.65      0.53      0.58       295

          micro avg       0.66      0.68      0.67      1928
          macro avg       0.65      0.65      0.64      1928
       weighted avg       0.67      0.68      0.67      1928
        samples avg       0.27      0.25      0.25      1928



In [43]:
import numpy as np

save_dir = "../DistilBERT/distilbert_crisis_final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
np.save(f"{save_dir}/best_thresholds.npy", best_t)